In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import matplotlib.pyplot as plt

In [5]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

text_english = open("tiny-shakespeare.txt").read()
text_turkish = open("tiny-shakespeare_tr.txt").read()

unique_chars_english = sorted(list(set(text_english)))
unique_chars_turkish = sorted(list(set(text_turkish)))

stoi_en = {ch:i for i, ch in enumerate(unique_chars_english)}
itos_en = {i:ch for i, ch in enumerate(unique_chars_english)}

stoi_tr = {ch:i for i, ch in enumerate(unique_chars_turkish)}
itos_tr = {i:ch for i, ch in enumerate(unique_chars_turkish)}

encode_en = lambda s: [stoi_en[ch] for ch in s]
decode_en = lambda ary: [itos_en[i] for i in ary]

encode_tr = lambda s: [stoi_tr[ch] for ch in s]
decode_tr = lambda ary: [itos_tr[i] for i in ary]

encoded_all_data_en = encode_en(text_english)
encoded_all_data_tensor_en = torch.tensor(encoded_all_data_en)
n_en = int(0.8 * len(encoded_all_data_tensor_en))
train_data_tensor_en = encoded_all_data_tensor_en[:n_en]
val_data_tensor_en = encoded_all_data_tensor_en[n_en:]

#send to GPU
train_data_tensor_en = train_data_tensor_en.to(device)
val_data_tensor_en = val_data_tensor_en.to(device)

# encoded_all_data_tr = encode_tr(text_turkish)
# encoded_all_data_tensor_tr = torch.tensor(encoded_all_data_tr)
# n_tr = int(0.8 * len(encoded_all_data_tensor_tr))
# train_data_tensor_tr = encoded_all_data_tensor_tr[:n_tr]
# val_data_tensor_tr = encoded_all_data_tensor_tr[n_tr:]
# train_data_tensor_tr = train_data_tensor_tr.to(device)
# val_data_tensor_tr = val_data_tensor_tr.to(device)

In [8]:
class AttentionHead(nn.Module):

    def __init__(self, num_embedding_C, head_size, masked=True) -> None:
        super().__init__()
        self.num_embedding_C = num_embedding_C
        self.head_size = head_size
        self.masked = masked
        # Query can come from a different context
        self.w_query = nn.Linear(in_features=self.num_embedding_C, out_features=self.head_size)
        # Key and Value should come from the same source
        self.w_key = nn.Linear(in_features=self.num_embedding_C, out_features=self.head_size)
        self.w_value = nn.Linear(in_features=self.num_embedding_C, out_features=self.head_size)

    def forward(self, source_x, target_context=None, key_padding_mask=None):
        if target_context is None:#self attention
            self.Q = self.w_query(source_x)
        else:
            self.Q = self.w_query(target_context) # cross-attention

        self.K = self.w_key(source_x)
        self.V = self.w_value(source_x)
        

class Translator(nn.Module):
    def __init__(self, vocab_size_en, vocab_size_tr, batch_size_B, block_size_T, num_embedding_C, attention_head_size_Hs, num_multi_attention_head_Hn, num_transformer_layer_Nx, dropout = 0.2, learning_rate=3e-4):
        super().__init__()
        self.vocab_size_en = vocab_size_en
        self.batch_size_B = batch_size_B
        self.block_size_T = block_size_T
        self.num_embedding_C = num_embedding_C
        self.attention_head_size_Hs = attention_head_size_Hs
        self.num_multi_attention_head_Hn = num_multi_attention_head_Hn
        self.num_transformer_layer_Nx = num_transformer_layer_Nx
        self.dropout = dropout
        self.learning_rate = learning_rate

        self.token_embeddings_en = nn.Embedding(num_embeddings=self.vocab_size_en, embedding_dim=self.num_embedding_C)
        self.position_embeddings_en = nn.Embedding(num_embeddings=self.block_size_T, embedding_dim=self.num_embedding_C)
        self.last_linear_head = nn.Linear(in_features=self.num_embedding_C, out_features=self.vocab_size_en)
        self.last_layernorm_head = nn.LayerNorm(self.num_embedding_C)

    def forward(self, X_idx_en, Y_targets):
        output_token_embedding = self.token_embeddings_en(X_idx_en) # [B, T, C]
        output_position_embedding = self.position_embeddings_en(torch.arange(self.block_size_T, device=device)) # [B, T, C]

        x = output_token_embedding + output_position_embedding
        # x = nn.Blocks(num_layer, head_size)
            # x = multiHeadAttention(x, num_layer, head_size)
                #SelfAttention
            # Add
            # layernorm
            # feed forward

        x = self.last_layernorm_head(x) # [B, T, C] - LayerNorm is shape-preserving. It normalises; it doesn't reduce T dimension.
        logits = self.last_linear_head(x) # [B, T, V = vocab_size]
        #flatten logits
        logits = logits.view(self.batch_size_B * self.block_size_T, self.num_embedding_C) #[B, T, C] --> [B*T, C]
        #flatten targets
        Y_targets = Y_targets.view(self.batch_size_B * self.block_size_T)
        #send to loss function
        loss = F.cross_entropy(logits, Y_targets) #[B*T, V] vs [B*T]

        return logits, loss








In [3]:
a = [i for i in range(8)]
b = torch.arange(8)

print(a)
print(b)

[0, 1, 2, 3, 4, 5, 6, 7]
tensor([0, 1, 2, 3, 4, 5, 6, 7])
